# RAG Chatbot with BioBERT and LLaMA 2

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline using biomedical embeddings (`BioBERT`) and a local language model (`LLaMA 2`) to answer domain-specific medical questions. The pipeline includes:

- Document loading and splitting  
- Embedding generation with `BioBERT`  
- Vector store creation using `Chroma`  
- Retrieval-based question answering with `LLaMA 2`  

This setup is ideal for medical QA tasks where factual grounding and domain relevance are critical.  
The model runs locally via `transformers` and Hugging Face pipelines, without requiring an API key.

---

In [ ]:
!pip install -U langchain langchain-community langchain-huggingface
!pip install -U chromadb
!pip install -U sentence-transformers
!pip install -U transformers
!pip install -U accelerate
!pip install -U langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00

In [ ]:
import os
import glob
import shutil

from huggingface_hub import login

from langchain.schema import Document
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

### Project Directory Connection


In [ ]:
!git clone https://github.com/a20190202/PLN_Medical_Flashcard.git

Cloning into 'PLN_Medical_Flashcard'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (283/283), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 283 (delta 96), reused 247 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (283/283), 38.29 MiB | 9.14 MiB/s, done.
Resolving deltas: 100% (96/96), done.
Updating files: 100% (76/76), done.


In [ ]:
!ls

PLN_Medical_Flashcard					     sample_data
pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb.zip


In [ ]:
%cd PLN_Medical_Flashcard/pln_model

/content/PLN_Medical_Flashcard/pln_model


In [ ]:
!pwd

/content/PLN_Medical_Flashcard/pln_model


# Vectorstore Generation
---

In [ ]:
def read_txt_files(folder_path):
    all_docs = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        length_function=len,
    )

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            chunks = splitter.split_text(text)

            for i, chunk in enumerate(chunks):
                doc = Document(
                    page_content=chunk,
                    metadata={
                        "source": filename,
                        "chunk_id": i,
                        "total_chunks": len(chunks)
                    }
                )
                all_docs.append(doc)

    return all_docs

all_documents = read_txt_files("data/textbooks")

## Embeddings model
### `pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb`

This SentenceTransformer model is a fine-tuned version of BioBERT on multiple natural language inference (NLI) and semantic similarity datasets, including:

- MNLI, SNLI, SciNLI, SciTail, MedNLI, and STS-B

It is specifically optimized for semantic similarity tasks in the biomedical domain and is suitable for generating high-quality dense vector embeddings of medical questions, terms, or documents.

- **Base model:** `dmis-lab/biobert-base-cased-v1.1`  
- **Embedding dimensions:** `768`  
- **Use case:** Biomedical sentence embeddings for retrieval and clustering

Model link: [https://huggingface.co/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb](https://huggingface.co/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb)


In [ ]:
EMBEDDINGS = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"

In [ ]:
embeddings_model = HuggingFaceEmbeddings(
        model_name=EMBEDDINGS,
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'FetchError: Could not fetch resource at https://colab.research.google.com/userdata/get?authuser=1&notebookid=1c89AwP1hJGBI-aVFgpHG91qUlwONavi4&key=HF_TOKEN: 401  '.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Create Vector Store (Run Only Once)
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings_model,
    persist_directory=f"./{EMBEDDINGS.replace('/','_')}"
)

### Save the vector store after creation

In [ ]:
# Define original path (correct one where Chroma actually saved the files)
vectorstore_dir = f"./{EMBEDDINGS.replace('/', '_')}"
zip_path = f"{vectorstore_dir}.zip"

# Create the ZIP file from the original directory
shutil.make_archive(vectorstore_dir, 'zip', vectorstore_dir)

# Move the ZIP to /content so it's visible in Colab file browser
!mv "{zip_path}" /content/

print(f"Vector store zipped and moved to /content/: {os.path.basename(zip_path)}")

✅ Vector store zipped and moved to /content/: pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb.zip


### Load the saved vector store

In [ ]:
# Unzip the saved vector store
vectorstore_dir = f"./{EMBEDDINGS.replace('/', '_')}"
zip_path = f"/content/{EMBEDDINGS.replace('/', '_')}.zip"

# Unzip only if not already extracted
if not os.path.exists(vectorstore_dir):
    shutil.unpack_archive(zip_path, vectorstore_dir)
    print(f"✅ Unzipped vector store to: {vectorstore_dir}")
else:
    print(f"ℹ️ Directory already exists: {vectorstore_dir}")

# Load the vector store
vectorstore = Chroma(
    persist_directory=vectorstore_dir,
    embedding_function=embeddings_model
)

✅ Unzipped vector store to: ./pritamdeka_BioBERT-mnli-snli-scinli-scitail-mednli-stsb


# RAG
---

In [ ]:
# OLlama download
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# Launch the Ollama server locally
!ollama serve > /dev/null 2>&1 &
!sleep 10

In [ ]:
!ollama pull phi3:latest
!pip install -U langchain-ollama

In [ ]:
from langchain_ollama import OllamaLLM

In [ ]:
llm = OllamaLLM(
    model="phi3:latest",
    temperature=0.2,
    system=""
)

## Personalized prompt:

In [ ]:
prompt = ChatPromptTemplate.from_template(
"""
Medical Flashcard Generator Prompt

You will receive the name of a medical condition or disease. Your task is to create 5 comprehensive flashcards that systematically cover the essential aspects of the condition for medical education purposes.

Required Coverage Areas:
1. Definition & Pathophysiology - Core concept and underlying mechanisms
2. Etiology & Risk Factors - Causes and predisposing factors
3. Clinical Presentation - Signs, symptoms, and clinical manifestations
4. Diagnostic Approach - Key tests, criteria, and differential considerations
5. Management & Treatment - Therapeutic interventions and prognosis

Flashcard Requirements:
- Each flashcard must contain one focused question and one comprehensive answer
- Questions should be clinically relevant and test practical knowledge
- Answers should be precise, direct, and medically accurate
- Provide specific details (lab values, medication dosages, timeframes where applicable)
- Use medical terminology appropriately while maintaining clarity
- Give concise, focused responses without bullet points or lists
- Prioritize high-yield information commonly tested in medical examinations

Context Considerations:
{context}

Output Format:
Flashcard 1: [Topic Area]
Q: [Specific, focused question]
A: [Precise, direct answer in paragraph form]

Flashcard 2: [Topic Area]
Q: [Specific, focused question]
A: [Precise, direct answer in paragraph form]

[Continue for all 5 flashcards]

Quality Standards:
- Ensure medical accuracy and evidence-based content
- Use current clinical guidelines and best practices
- Include relevant mnemonics or memory aids where helpful
- Maintain consistency in terminology and formatting
- Focus on clinically actionable information

Medical Condition: {input}
"""
 )

## RAG Pipeline:

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# RAG Chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

## Inference Test:

### __Diabetes:__

In [ ]:
question = "Diabetes"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1: Definition & Pathophysiology of Diabetes
Q: What is diabetes mellitus, and how does it differ in pathophysiology between type 1 and type 2?
A: Diabetes mellitus encompasses a group of metabolic disorders characterized by chronically elevated blood glucose levels due to defects in insulin secretion, action, or both. In Type 1 diabetes (T1D), the pathophysiology involves an autoimmune destruction leading to absolute deficiency of insulin production because pancreatic beta cells are mistakenly targeted by the immune system following a potential environmental trigger like viral infection, often with genetic predisposition. In contrast, Type 2 diabetes (T2D) is primarily due to an imbalance between insulin secretion and resistance of body tissues towards its action despite relative or absolute deficiency; it's commonly associated with obesity which exacerbates the condition through increased free fatty acids that impair beta-cell function.

Flashcard 2: Etiology & Ri

### __Asthma:__

In [ ]:
question = "Asthma"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1: Definition & Pathophysiology of Asthma
Q: What is asthma, and what are the primary pathological features observed in this chronic inflammatory disorder?
A: Asthma is a common long-term respiratory condition characterized by recurrent episodes of wheezing, breathlessness, chest tightness, and cough. These symptoms result from intermittent airway obstruction due to bronchial smooth muscle cell hypertrophy and hyperreactivity along with increased mucus secretion in the airways. The hallmark features include reversible airflow limitation during exacses of asthma, chronic inflammation predominantly involving eosinophils but also including neutrophils, lymphocytes, macrophages and epithelial cells, as well as bronchial hyperresponsiveness to various stimuli.

Flashcard 2: Etiology & Risk Factors for Asthma
Q: What are the major factors contributing to asthma development, particularly in genetically predisposed individuals?
A: The etiology of asthma is multifactorial a

### __Cardiac Arrest:__

In [ ]:
question = "Cardiac Arrest"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1 - Definition & Pathophysiology
Q: What is cardiac arrest, and how does it differ from a heart attack?
A: Cardiac arrest occurs when the heart's electrical system malfunctions, causing an abrupt cessation of effective blood circulation. Unlike myocardial infarction (heart attack), which results from blocked coronary arteries leading to tissue damage due to lack of oxygen, cardiac arrest is primarily a failure in the heart's electrical activity that halts pumping function without necessarily causing permanent organ damage if promptly treated.

Flashcard 2 - Etiology & Risk Factors
Q: What are common etiologies and risk factors for sudden cardiac death?
A: Sudden cardiac death often results from lethal arrhythmias such as ventricular fibrillation or tachycardia. Common causes include coronary artery disease, hypertension, heart failure, electrolyte imbalances like hypokalemia, and inherited conditions like long QT syndrome. Risk factors encompass age over 65 years, 

### __Gastritis:__

In [ ]:
question = "Gastritis"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1 - Definition & Pathophysiology
Q: What is gastritis, how does it differ from mucosal erythema seen during endoscopy, and what are the proposed mechanisms leading to its development?
A: Gastritis refers to histologically documented inflammation of the stomach lining or gastric mucosa. Unlike transient mucosal erythema that can be observed via endoscopy without underlying pathology, true gastritis involves cellular changes and damage detectable upon microscopic examination of biopsy samples from the affected area. The development of gastritis is multifactorial; however, one common etiological factor across various forms includes infection with Helicobacter pylori (H. pylori), which induces an inflammatory response that can lead to cellular damage and subsequent mucosal injury. Other mechanisms include autoimmune reactions as seen in atrophic gastritis, chemical irritation from substances like alcohol or bile acids leading to direct epithelial insults (gastropathy),

### __Stroke:__

In [ ]:
question = "Stroke"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

Respuesta: Flashcard 1: Definition & Pathophysiology of Stroke
Q: What is the definition of a stroke, including its pathophysiological mechanisms?
A: A stroke occurs when blood flow to an area of the brain is interrupted or reduced, depriving brain tissue of oxygen and nutrients. This interruption can be due to either blockage (ischemic stroke) caused by a clot within cerebral arteries or rupture (hemorrhagic stroke) leading to bleeding in the brain when blood vessels break open, often from high blood pressure. The pathophysiology involves neuronal death and inflammation due to ischemia if not promptly treated; hemorrhagic strokes can cause additional damage through increased intracranial pressure and direct tissue injury by blood products.

Flashcard 2: Etiology & Risk Factors for Stroke
Q: What are the primary etiological factors of stroke, along with associated risk factors?
A: The main causes of ischemic strokes include thrombosis within cerebral arteries due to atherosclerosis or 